# 03 — Parameter Reconstruction

This notebook performs the first inverse-modeling step.

It fits the non-ideal series RLC magnitude model to each resistance curve:

`R_total = R_measured + R_hidden`

The goal is not just to get fit values. The goal is to test whether recovered parameters are stable across resistance values.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.optimize import curve_fit

DATA_PATH = Path("../data/processed/resistance_sweep_combined_clean.csv")

PROCESSED_DIR = Path("../data/processed")
FIG_DIR = Path("../figures/parameter_reconstruction")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.grid": True,
})

In [ ]:
df = pd.read_csv(DATA_PATH)

required = ["Run", "Resistance", "Frequency_Hz", "Vs_Vpp", "VR_Vpp", "Gain"]
missing = [col for col in required if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}. Current columns: {df.columns.tolist()}")

for col in required:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=required)
df = df.sort_values(["Resistance", "Run", "Frequency_Hz"]).reset_index(drop=True)

df.head()

In [ ]:
def rlc_gain_model(f, R_measured, R_hidden, L, C):
    omega = 2 * np.pi * f
    R_total = R_measured + R_hidden
    X = omega * L - 1 / (omega * C)
    return R_measured / np.sqrt(R_total**2 + X**2)

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def resonance_frequency(L, C):
    return 1 / (2 * np.pi * np.sqrt(L * C))

In [ ]:
BOUNDS_LOWER = [0, 5e-3, 100e-9]     # R_hidden, L, C
BOUNDS_UPPER = [200, 20e-3, 400e-9]
INITIAL_GUESS = [40, 10e-3, 220e-9]

def fit_curve(data, label):
    R_measured = data["Resistance"].iloc[0]
    f = data["Frequency_Hz"].values
    y = data["Gain"].values

    def model_for_fit(f, R_hidden, L, C):
        return rlc_gain_model(f, R_measured, R_hidden, L, C)

    try:
        popt, pcov = curve_fit(
            model_for_fit,
            f,
            y,
            p0=INITIAL_GUESS,
            bounds=(BOUNDS_LOWER, BOUNDS_UPPER),
            maxfev=50000
        )

        R_hidden, L, C = popt
        y_fit = model_for_fit(f, R_hidden, L, C)

        return {
            "Label": label,
            "Resistance": R_measured,
            "R_hidden_Ohm": R_hidden,
            "L_eff_H": L,
            "L_eff_mH": L * 1e3,
            "C_eff_F": C,
            "C_eff_nF": C * 1e9,
            "f0_eff_Hz": resonance_frequency(L, C),
            "RMSE": rmse(y, y_fit),
            "Fit_Success": True,
            "Error": ""
        }

    except Exception as e:
        return {
            "Label": label,
            "Resistance": R_measured,
            "R_hidden_Ohm": np.nan,
            "L_eff_H": np.nan,
            "L_eff_mH": np.nan,
            "C_eff_F": np.nan,
            "C_eff_nF": np.nan,
            "f0_eff_Hz": np.nan,
            "RMSE": np.nan,
            "Fit_Success": False,
            "Error": str(e)
        }

In [ ]:
# Fit Run 1 and Run 2 separately.
by_run_results = []

for (run, R), group in df.groupby(["Run", "Resistance"]):
    by_run_results.append(fit_curve(group, label=f"Run {int(run)}"))

fit_by_run = pd.DataFrame(by_run_results).sort_values(["Resistance", "Label"])
fit_by_run.to_csv(PROCESSED_DIR / "parameter_reconstruction_by_run.csv", index=False)

fit_by_run

In [ ]:
# Fit combined Run 1 + Run 2 for each resistance.
combined_results = []

for R, group in df.groupby("Resistance"):
    combined_results.append(fit_curve(group, label="Combined"))

fit_combined = pd.DataFrame(combined_results).sort_values("Resistance")
fit_combined.to_csv(PROCESSED_DIR / "parameter_reconstruction_combined.csv", index=False)

fit_combined

In [ ]:
for R, group in df.groupby("Resistance"):
    group = group.sort_values("Frequency_Hz")
    row = fit_combined[fit_combined["Resistance"] == R].iloc[0]

    if not row["Fit_Success"]:
        continue

    R_hidden = row["R_hidden_Ohm"]
    L_eff = row["L_eff_H"]
    C_eff = row["C_eff_F"]

    f_smooth = np.linspace(group["Frequency_Hz"].min(), group["Frequency_Hz"].max(), 2000)
    y_smooth = rlc_gain_model(f_smooth, R, R_hidden, L_eff, C_eff)

    fig, ax = plt.subplots(figsize=(9, 5))

    for run, run_group in group.groupby("Run"):
        ax.semilogx(run_group["Frequency_Hz"], run_group["Gain"], "o", markersize=4, label=f"Run {int(run)}")

    ax.semilogx(f_smooth, y_smooth, "-", linewidth=2, label="Combined fitted model")

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Gain |VR/Vs|")
    ax.set_title(f"Parameter Reconstruction Fit, R = {R:.0f} Ω")
    ax.grid(True, which="both", alpha=0.35)
    ax.legend()

    text = (
        f"R_hidden = {R_hidden:.2f} Ω\n"
        f"L_eff = {L_eff*1e3:.2f} mH\n"
        f"C_eff = {C_eff*1e9:.1f} nF\n"
        f"RMSE = {row['RMSE']:.4f}"
    )

    ax.text(0.03, 0.97, text, transform=ax.transAxes, va="top",
            bbox=dict(boxstyle="round", alpha=0.15))

    fig.savefig(FIG_DIR / f"fit_vs_data_R_{int(R)}ohm.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
for R, group in df.groupby("Resistance"):
    group = group.sort_values("Frequency_Hz")
    row = fit_combined[fit_combined["Resistance"] == R].iloc[0]

    if not row["Fit_Success"]:
        continue

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.axhline(0, linestyle="--", linewidth=1)

    for run, run_group in group.groupby("Run"):
        y_fit = rlc_gain_model(
            run_group["Frequency_Hz"].values,
            R,
            row["R_hidden_Ohm"],
            row["L_eff_H"],
            row["C_eff_F"]
        )
        residual = run_group["Gain"].values - y_fit

        ax.semilogx(run_group["Frequency_Hz"], residual, "o-", markersize=4, label=f"Run {int(run)}")

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Residual")
    ax.set_title(f"Residuals, R = {R:.0f} Ω")
    ax.grid(True, which="both", alpha=0.35)
    ax.legend()

    fig.savefig(FIG_DIR / f"residuals_R_{int(R)}ohm.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
reconstruction_summary = fit_combined[
    ["Resistance", "R_hidden_Ohm", "L_eff_mH", "C_eff_nF", "RMSE"]
].copy()

reconstruction_summary.to_csv(PROCESSED_DIR / "reconstruction_summary.csv", index=False)

reconstruction_summary

In [ ]:
def save_line_plot(x, y, xlabel, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x, y, "o-", linewidth=2)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.35)

    fig.savefig(FIG_DIR / f"{filename}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{filename}.svg", bbox_inches="tight")
    plt.show()

save_line_plot(reconstruction_summary["Resistance"], reconstruction_summary["R_hidden_Ohm"],
               "Measured Resistance R (Ω)", "Recovered Hidden Resistance (Ω)",
               "Recovered Hidden Resistance vs Measured Resistance", "R_hidden_vs_measured_R")

save_line_plot(reconstruction_summary["Resistance"], reconstruction_summary["L_eff_mH"],
               "Measured Resistance R (Ω)", "Recovered Effective Inductance (mH)",
               "Recovered Effective Inductance vs Measured Resistance", "L_eff_vs_measured_R")

save_line_plot(reconstruction_summary["Resistance"], reconstruction_summary["C_eff_nF"],
               "Measured Resistance R (Ω)", "Recovered Effective Capacitance (nF)",
               "Recovered Effective Capacitance vs Measured Resistance", "C_eff_vs_measured_R")

save_line_plot(reconstruction_summary["Resistance"], reconstruction_summary["RMSE"],
               "Measured Resistance R (Ω)", "RMSE",
               "Model Error vs Measured Resistance", "RMSE_vs_measured_R")

## Interpretation

If the model were perfectly physical and identifiable, recovered `L_eff`, `C_eff`, and `R_hidden` would be roughly invariant across measured resistance.

If they vary systematically, the fitted parameters should be interpreted as **effective parameters**, not direct component measurements.

Low RMSE means the curve was reproduced well. It does not prove that the recovered parameters are unique or physically true.